# Phase 2B training on Kaggle GPU

Trains the Phase 2B models with the **same protocol as the local runs**: same LOBO folds, splits, windows, loss, optimizer, schedule and early stopping (`configs/graph_dynamics.json`, `configs/dynamics_gru.json`). Only the device differs; it is recorded in every `train_log*.json`.

- graph-aware model: 6 folds, seed 0 → `results/phase2b/graph_dynamics/fold_*/graph_seed0.pt`
- GRU ensemble members: seeds 1–2 for folds 0–4 → `results/phase2b/ensemble/gru/fold_*/gru_seed{1,2}.pt`. Fold 5 seeds 1–2 already exist in `results/phase2a`.

**Settings:** Accelerator = GPU (T4 x2 or P100), Internet = on.

**Data:** attach a Kaggle Dataset containing `phase1_sample.zip` (made locally from `data/phase1_sample`), or let cell 4 regenerate it deterministically (~40–60 min).

**Output:** download `/kaggle/working/phase2b_kaggle_checkpoints.zip` and unzip it at the repo root.

In [ ]:
BRANCH = "phase2b"          # git branch with the Phase 2B code
STARTER_KIT_REV = "1fb75606d12e016463fbc63de704967c5b2bf9d5"   # revision the Phase 1 data was collected with
PARALLEL = 3                # concurrent training processes (Kaggle has 4 CPU cores; data loading is CPU-bound)
WORKERS = 1                 # DataLoader workers per process
THREADS = 1                 # torch CPU threads per process
GRAPH_JOBS = "0:0 1:0 2:0 3:0 4:0 5:0"
GRU_JOBS = "0:1 1:1 2:1 3:1 4:1 0:2 1:2 2:2 3:2 4:2"
REPO = "/kaggle/working/smart-building-rl"
PY = "/kaggle/working/venv/bin/python"


In [ ]:
!git clone -b {BRANCH} https://github.com/fannam/smart-building-rl.git {REPO}
!pip install -q uv
!uv venv --python 3.12 /kaggle/working/venv
!git clone https://github.com/smart-buildings-2026/starter-kit.git {REPO}/starter-kit && cd {REPO}/starter-kit && git checkout -q {STARTER_KIT_REV}
!uv pip install -p {PY} -e {REPO}/starter-kit
!uv pip install -p {PY} "numpy==2.0.2" "pandas==2.3.3" "torch>=2.4" scikit-learn
!{PY} -c "import torch; print(torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"


In [ ]:
# Data: use an attached dataset if present, otherwise regenerate deterministically (same argv as the original collection).
import glob, os, subprocess
os.makedirs(f"{REPO}/data", exist_ok=True)
zips = glob.glob("/kaggle/input/**/phase1_sample.zip", recursive=True)
dirs = [d for d in glob.glob("/kaggle/input/**/phase1_sample", recursive=True) if os.path.isdir(d)]
if os.path.exists(f"{REPO}/data/phase1_sample/manifest.jsonl"):
    print("data present")
elif dirs:
    os.symlink(dirs[0], f"{REPO}/data/phase1_sample"); print("linked", dirs[0])
elif zips:
    subprocess.run(["unzip", "-q", zips[0], "-d", f"{REPO}/data"], check=True); print("unzipped", zips[0])
else:
    print("regenerating data/phase1_sample (deterministic, ~40-60 min) ...")
    subprocess.run(f"cd {REPO} && MPLBACKEND=Agg {PY} -u src/collect_data.py --out data/phase1_sample --profiles baseline moderate "
                   f"--episodes 1 --steps 2016 --seed 0 --workers 4", shell=True, check=True)
n = sum(1 for _ in open(f"{REPO}/data/phase1_sample/manifest.jsonl"))
assert n == 60, n
print("episodes:", n)


In [ ]:
# GPU smoke test: 1 epoch on a few windows for both models. Stops here if the CUDA path is broken.
import json
for src, dst in [("configs/graph_dynamics.json", "/kaggle/working/smoke_graph.json"), ("configs/dynamics_gru.json", "/kaggle/working/smoke_gru.json")]:
    c = json.load(open(f"{REPO}/{src}")); c["training"].update(epochs=1, max_train_windows=64, val_stride=60, min_epochs=0)
    json.dump(c, open(dst, "w"), indent=1)
!cd {REPO} && {PY} scripts/train_graph.py --config /kaggle/working/smoke_graph.json --jobs 5:0 --parallel 1 --out /kaggle/working/smoke/graph --device cuda --workers {WORKERS} --threads {THREADS}
!cd {REPO} && {PY} scripts/train_dynamics.py --config /kaggle/working/smoke_gru.json --jobs 5:9 --parallel 1 --out /kaggle/working/smoke/gru --device cuda --workers {WORKERS} --threads {THREADS}
for f in ["/kaggle/working/smoke/graph/fold_5_building_train_8/train_log_graph_seed0.json", "/kaggle/working/smoke/gru/fold_5_building_train_8/train_log_seed9.json"]:
    L = json.load(open(f)); print(f.split('/')[-1], L["device"], "epoch_s", L["history"][0]["epoch_s"]); assert L["device"].startswith("cuda")


In [ ]:
# Graph-aware model, 6 LOBO folds (seed 0)
!cd {REPO} && {PY} -u scripts/train_graph.py --jobs {GRAPH_JOBS} --parallel {PARALLEL} --device cuda --workers {WORKERS} --threads {THREADS} 2>&1 | grep -v Warning
!tail -n 3 {REPO}/results/phase2b/graph_dynamics/fold_*/train_graph_0.log


In [ ]:
# GRU ensemble members (seeds 1-2, folds 0-4)
!cd {REPO} && {PY} -u scripts/train_dynamics.py --config configs/dynamics_gru.json --jobs {GRU_JOBS} --parallel {PARALLEL} --out results/phase2b/ensemble/gru --device cuda --workers {WORKERS} --threads {THREADS} 2>&1 | grep -v Warning


In [ ]:
# Package checkpoints + logs for download (unzip at the repo root locally)
!cd {REPO} && zip -qr /kaggle/working/phase2b_kaggle_checkpoints.zip results/phase2b/graph_dynamics results/phase2b/ensemble -x "*.npz"
!ls -lh /kaggle/working/phase2b_kaggle_checkpoints.zip
